<a href="https://colab.research.google.com/github/ezpzzbeh/data_science_toolkit/blob/main/Salaries_Distribution_Tutorial6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Pandas let us work with spreadsheet-like tables.
import pandas as pd

#These tools scale numerical data for machine learning.
from sklearn.preprocessing import MinMaxScaler, StandardScaler

#Read both files into separate dataframes. today, got 2 sets of data.
df_salary = pd.read_csv("/content/drive/MyDrive/salary.csv")
df_xtra = pd.read_csv("/content/drive/MyDrive/xtra_info.csv")


#Print the number of rows and columns in each table
print("Salary table size:",df_salary.shape)
print("Extra table size:", df_xtra.shape)

#show a small preview of each dataset
display(df_salary.head(3))
display(df_xtra.head(3))

Salary table size: (28180, 19)
Extra table size: (13404, 4)


,Unnamed: 0,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,0,2021-04-27 11:02:09.743,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,55000,0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,1,2021-04-27 11:02:21.562,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,54600,4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,2,2021-04-27 11:02:38.125,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,34000,NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White


,USERID,CGPA,Salary_month_USD,Special Bonus_GBP
0,8683,2.990627,4583.333333,2437.0
1,3880,2.288131,5166.666667,740.0
2,8898,3.903324,5000.000000,3806.0


**1.Shorten long column names using column numbers**
Pandas count column from 0. so df_salary.columns[6] means 7th column.

In [3]:
#Build a rename dictionary using COLUMN POSITIONS (not long names)
short_names = {
    df_salary.columns[0]: "USERID",
    df_salary.columns[6]: "Annual_Salary",
    df_salary.columns[7]: "Extra_Compensation",
    df_salary.columns[8]: "Currency",
    df_salary.columns[11]: "Country",
    df_salary.columns[13]: "City"
}

#Apply these shorter names to a fresh copy of our salary data.
df_salary = df_salary.rename(columns = short_names)

#make sure IDs and numerical data are stored as numbers.
df_salary["USERID"] = pd.to_numeric(df_salary["USERID"], errors = "coerce")
df_salary["Annual_Salary"] = pd.to_numeric(
    df_salary["Annual_Salary"].astype(str).str.replace(",","", regex=False),  #replace , with blank exp:55,000 to 55000 pure numbers
    errors = "coerce"
)
df_xtra["USERID"] = pd.to_numeric(df_xtra["USERID"], errors = "coerce")

#Display only the columns we need for this class
display(df_salary[["USERID","Annual_Salary","Currency","Country","City"]].head())

,USERID,Annual_Salary,Currency,Country,City
0,0,55000,USD,United States,Boston
1,1,54600,GBP,United Kingdom,Cambridge
2,2,34000,USD,US,Chattanooga
3,3,62000,USD,USA,Milwaukee
4,4,60000,USD,US,Greenville


# **1. Data Integration**
Do we use pd.merge or concat()?
ANS: pd.merge! because both file have shared USERID. concat() normally stacks table beside each other on top of each other, doesn't automaticallymatch people by ID.


check the xtra file may contain repeated ID.Keep only 1.

In [4]:
#Count the total number of extra rows and how many IDs are different.
print("Extra rows:", len(df_xtra))
print("Unique extra USERIDs:",df_xtra["USERID"].nunique())#unique means not repeated

#Count all rows that repeat an earlier USERID
print("Repeated USERID rows:", df_xtra.duplicated(subset ="USERID").sum())

#THIS IS NOT REAL LIFE DECISION: keep 1st row per USERID,get rid others
df_xtra_unique = df_xtra.drop_duplicates(subset ="USERID",keep="first").copy()

#Check every USERID now occurs at most once
print("After removing rpeated IDs:", len(df_xtra_unique))
assert not df_xtra_unique["USERID"].duplicated().any()


Extra rows: 13404
Unique extra USERIDs: 8442
Repeated USERID rows: 4962
After removing rpeated IDs: 8442


# **Merge 2 data frames**

In [5]:
#Merge extra colums onto the original salary rows using USERID
df_integrated = pd.merge(
    df_salary,
    df_xtra_unique,
    on="USERID", # keep records taht have same ID
    how="left",  #keep every row in ori salary table
    validate="one_to_one"  #check IDs are unique in both tables
)

#A safe left merge should not create extra salary rows.(SAME LEN)
print("Salary rows before merge:", len(df_salary))
print("Salary rows after merge:", len(df_integrated))
#check how many rows received extra info to do this , check how many r not null
print("Rows with extra info:", df_integrated["CGPA"].notna().sum())

#assert check idf condition is true.
# check if both have the same num of rows to make sure no salary rows were added or lost
assert len(df_integrated) == len(df_salary)

#Look at merged columns
display(df_integrated[["USERID","Annual_Salary","Currency","CGPA","Salary_month_USD","Special Bonus_GBP"]].head(8))

Salary rows before merge: 28180
Salary rows after merge: 28180
Rows with extra info: 8442


,USERID,Annual_Salary,Currency,CGPA,Salary_month_USD,Special Bonus_GBP
0,0,55000,USD,NaN,NaN,NaN
1,1,54600,GBP,2.695615,4250.000000,125.0
2,2,34000,USD,NaN,NaN,NaN
3,3,62000,USD,NaN,NaN,NaN
4,4,60000,USD,2.689383,12500.000000,1133.0
5,5,62000,USD,NaN,NaN,NaN
6,6,33000,USD,3.541300,6333.333333,878.0
7,7,50000,USD,NaN,NaN,NaN


Check whether the monthly salary from xtra file is derivable.

Remove derivable column,something we can calculate from existing other values.

In [6]:
#COMPARE MONTHLY SALARY FIGURE WHEN ORI SALARY CURRENCY IS USD.
#using & means if both conditions are true then selected theses 3 columns
comparison=df_integrated.loc[
    (df_integrated["Currency"]=="USD")& # chceck == if it is equal
    df_integrated["Salary_month_USD"].notna(),#not empty
    ["USERID","Annual_Salary","Salary_month_USD"]
].copy()#id=f both are true, copy columns store in comparison

#Calculate what monthly salary would be from annual USD salary
comparison["Caluculated_MOnthly_USD"] = comparison["Annual_Salary"]/12

#Check roughly how many supplied figures agree(allow tiny rounding differences)
rough_match = (
    comparison["Salary_month_USD"] - comparison["Caluculated_MOnthly_USD"]
).abs()<0.01
print("Number of comparable records:", len(comparison))
print("Matching monthly amounts:", rough_match.sum())
print("Nonmatching monthly amounts:",(~rough_match).sum())

display(comparison.head())

Number of comparable records: 7063
Matching monthly amounts: 55
Nonmatching monthly amounts: 7008


,USERID,Annual_Salary,Salary_month_USD,Caluculated_MOnthly_USD
4,4,60000,12500.000000,5000.000000
6,6,33000,6333.333333,2750.000000
9,9,45000,12500.000000,3750.000000
10,10,47500,4583.333333,3958.333333
12,12,100000,12083.333330,8333.333333


# **Drop derivable data**

In [ ]:
df_integrated["Annual_Salary_Div_12"] = df_integrated["Annual_Salary"]/12

df_integrated = df_integrated.drop(columns=["Annual_Salary_Div_12"])

#confirm it has been removed
print("Derived example column removed:",
      "Annual_Salary_Dix_12" not in df_integrated.columns)



Derived example column removed: True


# **Convert Special Bonus_GBP to USD**

In [ ]:
#assumption
GBP_TO_USD = 1.30

#new column converted bonus(USD)
df_integrated["Special_Bonus_USD"] = (
    pd.to_numeric(df_integrated["Special Bonus_GBP"],errors= "coerce")
    *GBP_TO_USD
)
#errors=coerce means"if a value can't converted to a number, replace with NaN missing value"

#Drop old
df_integrated = df_integrated.drop(columns=["Special Bonus_GBP"])

display(df_integrated.loc[
    df_integrated["Special_Bonus_USD"].notna(),
    ["USERID","Special_Bonus_USD"]
].head())

KeyError: 'Special Bonus_GBP'

# **2.Data Transformation-Normalization**

Machine-learning metjods sometimes work better than numeric inputs have similar scales.
-Min-Max puts between 0 and 1
-Z-score / StandardScaler changes vlaues so the fitted data have mean near 0 and standard deviation near 1

In [7]:
#Select only salaries reported in USD
df_usd = df_integrated.loc[df_integrated["Currency"]=="USD"].copy()

#keep only sensible,numeric,positive salaries
df_usd = df_usd.loc[
    df_usd["Annual_Salary"].notna()&(df_usd["Annual_Salary"]>0)
].copy()

#print("USD salary records:", len(df_usd))
display(df_usd[["Annual_Salary"]].head())

,Annual_Salary
0,55000
2,34000
3,62000
4,60000
5,62000


Option1
Min-max normalization
Formula: (value-min)/(max-min)

## *#although many 00 caming out which is more messy,but is is suitable for machine to learn*

In [11]:
minmax = MinMaxScaler()

#fit-transform learns range and scales column to  0-1
df_usd["Salary_MinMax"]=minmax.fit_transform(df_usd[["Annual_Salary"]]).ravel()
#ravel convert result from 2D array into 1D array
#from[[0.0],[0.5]] to [0.0,0.5]
#compare ori salaru with scaled value/
display(df_usd[["Annual_Salary","Salary_MinMax"]].head(10))
print("minimum scaled salary:",df_usd["Salary_MinMax"].min())
print("maximum scaled salary:",df_usd["Salary_MinMax"].max())

,Annual_Salary,Salary_MinMax
0,55000,0.000539
2,34000,0.000333
3,62000,0.000608
4,60000,0.000588
5,62000,0.000608
6,33000,0.000324
7,50000,0.000490
8,112000,0.001098
9,45000,0.000441
10,47500,0.000466


minimum scaled salary: 0.0
maximum scaled salary: 1.0


Option2
Z-score
Formula:(value-min)/Standard Deviation

Z-score of
0 :average
positive:above avg
negative:below avg
Z-score not restricted to 0 and 1, unllike MInMAX

In [12]:
#StandardScaler learns mean and SD
zscore_scaler = StandardScaler()

df_usd["Salary_Zscore"] = zscore_scaler.fit_transform( #new column []Salary_zcsore
    df_usd[["Annual_Salary"]]#transform this column
).ravel()

#compare ori salary with both scaling result
display(df_usd[["Annual_Salary","Salary_MinMax","Salary_Zscore"]].head(10))

#calculate avg standardized salaries
print("Mean:",df_usd["Salary_Zscore"].mean())

#calculate how spread out the standardized salaries aare
print("standard deviation:",df_usd["Salary_Zscore"].std())

,Annual_Salary,Salary_MinMax,Salary_Zscore
0,55000,0.000539,-0.058724
2,34000,0.000333,-0.086545
3,62000,0.000608,-0.049450
4,60000,0.000588,-0.052100
5,62000,0.000608,-0.049450
6,33000,0.000324,-0.087870
7,50000,0.000490,-0.065348
8,112000,0.001098,0.016792
9,45000,0.000441,-0.071972
10,47500,0.000466,-0.068660


Mean: -9.699828311216759e-18
standard deviation: 1.0000213308305577


# **3.Data Reduction - Aggregation and Grouping**

*Tutorial ask group salaries by city*

In [13]:
#remove rows with missing/empty city names.
df_city = df_usd.dropna(subset = ["City"]) #store in new dataset df_city

#same country for rows, same city in group
city_summary = df_city.groupby(["Country","City"],as_index=False).agg( #groupby collects records n same countrt/city combination
    Number_of_Records=("Annual_Salary","count"),#how many reported salaries  #agg calculate different summary statistics for eac group
    Mean_Salary_USD=("Annual_Salary","mean"),
    Median_Salary_USD=("Annual_Salary","median"), #infront black colours are all COLUMN NAMES
    Minimum_Salary_USD=("Annual_Salary","min"),
    Maximum_Salary_USD=("Annual_Salary","max")
)

display(city_summary.head(10))#display some exp

,Country,City,Number_of_Records,Mean_Salary_USD,Median_Salary_USD,Minimum_Salary_USD,Maximum_Salary_USD
0,U.S.,indianapolis,1,42500.0,42500.0,42500,42500
1,US,Philadelphia,1,86000.0,86000.0,86000,86000
2,United States,South Bend,1,69000.0,69000.0,69000,69000
3,Africa,Africa,1,90000.0,90000.0,90000,90000
4,America,Ann Arbor,1,34000.0,34000.0,34000,34000
5,America,Atlanta,1,45000.0,45000.0,45000,45000
6,America,Boston,1,194000.0,194000.0,194000,194000
7,America,Chicago,1,70.0,70.0,70,70
8,America,Columbia,1,75000.0,75000.0,75000,75000
9,America,Houston,1,20488.0,20488.0,20488,20488


*Sort cities from highest mean salary to lowest*

In [16]:
#ascending = False means Highest First

all_cities_sorted = city_summary.sort_values(
    by="Mean_Salary_USD",ascending=False
)

print("Top 10 city/country groups(all sizes):")
display(all_cities_sorted.head(10))

Top 10 city/country groups(all sizes):


,Country,City,Number_of_Records,Mean_Salary_USD,Median_Salary_USD,Minimum_Salary_USD,Maximum_Salary_USD
62,Colombia,Bogota,2,51012500.0,51012500.0,25000,102000000
64,Colombia,Cali,2,26106000.0,26106000.0,12000,52200000
6924,usa,Rice,1,10000000.0,10000000.0,10000000,10000000
6609,Usa,Pinebrook,1,5000044.0,5000044.0,5000044,5000044
59,China,Hong Kong,1,5000000.0,5000000.0,5000000,5000000
2524,USA,Newyork,1,2600000.0,2600000.0,2600000,2600000
175,Taiwan,Milwaukee,1,2000000.0,2000000.0,2000000,2000000
102,India,Banglore,1,1000000.0,1000000.0,1000000,1000000
3639,United Stated,Spokane,1,832000.0,832000.0,832000,832000
430,U.S.,Salina,1,734400.0,734400.0,734400,734400
